In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

2024-05-20 02:31:15.275668: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-20 02:31:15.275721: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-20 02:31:15.276528: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-20 02:31:15.281765: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-20 02:31:15.876431: W tensorflow/compiler/tf2

In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

test_data = test_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-20 02:31:17.731921: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 02:31:17.756151: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 02:31:17.756194: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 02:31:17.758497: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-20 02:31:17.758550: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [3]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def HModel(hp):
    model = keras.Sequential([
        
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
       
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        
        keras.layers.Flatten(),
        keras.layers.Dropout(hp.Float("dropout", min_value=0.2, max_value=0.4, sampling="log")),
        keras.layers.Dense(units=256, activation="relu" ),
        keras.layers.Dense(10, activation='softmax',activity_regularizer = tf.keras.regularizers.L1(hp.Float("regularization", min_value=0.007, max_value=0.0075, sampling="log"))),
    ])
    #hp.Float("dropout", min_value=0.2, max_value=0.25, sampling="log")
    
    #activity_regularizer = tf.keras.regularizers.L1(hp.Float("regularization", min_value=0.0066, max_value=0.008, sampling="log"))
    #units=hp.Int("units", min_value=224, max_value=288, step=32)
    
    #learning_rate = hp.Float("lr", min_value=0.0019, max_value=0.0022, sampling="log")
    #min_value=1e-4, max_value=1e-2 0.0001 0.01
    learning_rate = hp.Float("lr", min_value=0.0019, max_value=0.0022, sampling="log")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
        metrics=["accuracy"],
    )
    return model

HModel(kt.HyperParameters())

tuner = kt.RandomSearch(
    hypermodel=HModel,
    objective="val_accuracy",
    max_trials=3,
    executions_per_trial=2,
    overwrite=True,
    directory="./ACML_Project",
    project_name="ACML",
)
tuner.search_space_summary()


tuner.search(train_data, epochs=5, validation_data=val_data)

models = tuner.get_best_models(num_models=2)
best_model = models[0]
best_model.summary()

#Retraining the model with the best hyperparamters
#Top 2
best = tuner.get_best_hyperparameters(5)

#Building model
model = HModel(best[0])
print("best learning rate:"+str(best[0].get("lr")))
#print("best num units:"+str(best[0].get("units")))
print("best drop out:"+str(best[0].get("dropout")))
print("best regularization:"+str(best[0].get("regularization")))

earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True)
#
#
history = model.fit(
train_data,
epochs = 200,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping])

#best learning rate:0.002113314496627678
#best regularization:0.007189638437333526

Trial 3 Complete [00h 00m 44s]
val_accuracy: 0.25

Best val_accuracy So Far: 0.3799999952316284
Total elapsed time: 00h 02m 17s
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 batch_normalization (Batch  (None, 109, 109, 64)      256       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 109, 109, 64)      0    

In [7]:
model.evaluate(val_data, return_dict = True)

2/2 [==============================] - 0s 38ms/step - loss: 1.1161 - accuracy: 0.7200


{'loss': 1.116073727607727, 'accuracy': 0.7200000286102295}

In [5]:
model.evaluate(test_data, return_dict = True)

2/2 [==============================] - 0s 18ms/step - loss: 2.3030 - accuracy: 0.1400


{'loss': 2.3030354976654053, 'accuracy': 0.14000000059604645}

In [6]:
#model.save(filepath = "model_1.keras")